# RAG & Agent Evaluation — LangGraph Walkthrough

This notebook implements the evaluation concepts as **runnable LangGraph pipelines**, using mock tools/judges so everything runs with no API key.

**Structure:**
1. Part 1 — RAG retrieval metrics (Precision@K, Recall@K, MRR, nDCG)
2. Part 2 — RAG eval pipeline as a LangGraph graph (context relevance + faithfulness/groundedness)
3. Part 3 — Agent eval pipeline as a LangGraph graph (tool selection, task completion, trajectory scoring, unnecessary tool calls)

Each section starts with the interview question it answers, then the code that demonstrates it.

## Setup

```
pip install langgraph langchain-core
```

In [1]:
from typing import TypedDict, List, Dict, Optional
import math
from langgraph.graph import StateGraph, END

print("Imports OK")

Imports OK


---
# Part 1 — Retrieval Metrics

**Interview questions covered:**
- *How would you evaluate retrieval quality?*
- *What do Precision@K and Recall@K tell you?*
- *When would you use MRR or nDCG?*

**Approach:** build a labeled eval set (query → ground-truth relevant chunk IDs), run retrieval, score with rank-aware metrics — kept separate from generation quality so you can isolate *where* a RAG failure comes from.

In [2]:
def precision_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    """Of the K chunks retrieved, how many are actually relevant?"""
    top_k = retrieved[:k]
    hits = [doc for doc in top_k if doc in relevant]
    return len(hits) / k if k > 0 else 0.0


def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    """Of all relevant chunks that exist, how many did we find in top K?"""
    top_k = retrieved[:k]
    hits = [doc for doc in top_k if doc in relevant]
    return len(hits) / len(relevant) if relevant else 0.0


def mrr(retrieved: List[str], relevant: List[str]) -> float:
    """Mean Reciprocal Rank: 1/rank of the FIRST relevant hit.
    Use when there's typically one correct/best chunk and you care how early it appears."""
    for rank, doc in enumerate(retrieved, start=1):
        if doc in relevant:
            return 1.0 / rank
    return 0.0


def dcg_at_k(retrieved: List[str], grades: Dict[str, int], k: int) -> float:
    score = 0.0
    for i, doc in enumerate(retrieved[:k]):
        rel = grades.get(doc, 0)
        score += rel / math.log2(i + 2)  # rank 1 -> log2(2)=1, avoids log2(1)=0 division
    return score


def ndcg_at_k(retrieved: List[str], grades: Dict[str, int], k: int) -> float:
    """Normalized DCG: use when relevance is GRADED (not binary) and multiple
    relevant chunks can coexist. Rewards relevant results appearing higher."""
    dcg = dcg_at_k(retrieved, grades, k)
    ideal_order = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_order))
    return dcg / idcg if idcg > 0 else 0.0

print("Metric functions defined")

Metric functions defined


### Toy eval example

Query: *"What is the time complexity of BST delete?"*
- `retrieved_ids` — what the retriever actually returned, in rank order
- `relevant_ids` — ground truth (labeled once, reused across eval runs)
- `relevance_grades` — graded relevance (0-3) for nDCG; binary membership in `relevant_ids` is enough for Precision/Recall/MRR

In [3]:
query = "What is the time complexity of BST delete?"

retrieved_ids = ["chunk_3", "chunk_1", "chunk_7", "chunk_2", "chunk_9"]
relevant_ids  = ["chunk_1", "chunk_2", "chunk_5"]     # chunk_5 exists but wasn't retrieved -> hurts recall
relevance_grades = {
    "chunk_1": 3, "chunk_2": 2, "chunk_3": 0,
    "chunk_5": 3, "chunk_7": 1, "chunk_9": 0,
}

k = 5
print(f"Precision@{k}: {precision_at_k(retrieved_ids, relevant_ids, k):.2f}")
print(f"Recall@{k}:    {recall_at_k(retrieved_ids, relevant_ids, k):.2f}")
print(f"MRR:           {mrr(retrieved_ids, relevant_ids):.2f}")
print(f"nDCG@{k}:       {ndcg_at_k(retrieved_ids, relevance_grades, k):.2f}")

Precision@5: 0.40
Recall@5:    0.67
MRR:           0.50
nDCG@5:       0.51


**Reading the output:**
- Precision@5 = 0.40 → 2 of the 5 retrieved chunks were relevant (`chunk_1`, `chunk_2`). 3 were noise.
- Recall@5 = 0.67 → we found 2 of the 3 relevant chunks that exist; `chunk_5` was missed entirely.
- MRR = 0.50 → the first relevant chunk (`chunk_1`) appeared at rank 2, so 1/2.
- nDCG@5 factors in that `chunk_1` (grade 3) ranking below `chunk_3` (grade 0) is a bigger penalty than a small rank swap between two similarly-graded chunks would be.

---
# Part 2 — RAG Eval Pipeline as a LangGraph Graph

**Interview questions covered:**
- *How would you measure context relevance?*
- *How would you know the answer is grounded in the retrieved context?*

**Approach:** model the pipeline as a graph — `retrieve → judge_context_relevance → generate → check_faithfulness`. Each node writes eval signals into shared state. In production the "judge" functions below would be LLM calls (LLM-as-judge); here they're deterministic mocks so the notebook runs standalone.

- **Context relevance** = are the retrieved chunks actually about the query? (checked *before* generation)
- **Faithfulness/groundedness** = does the generated answer only assert things the context supports? (checked *after* generation, catches hallucination)

In [4]:
class RAGEvalState(TypedDict):
    query: str
    retrieved_chunks: List[Dict]
    relevance_judgments: Dict[str, str]     # chunk_id -> "relevant" / "irrelevant"
    answer: str
    claims: List[str]
    faithfulness_scores: Dict[str, bool]    # claim -> is it supported by context?
    context_relevance_score: float
    faithfulness_score: float

In [5]:
# ---- Mock "LLM-as-judge" functions ----
# In production: replace these with real LLM calls (e.g. "Is this chunk relevant to
# the query? yes/no", or "Is this claim entailed by the context? yes/no").

STOPWORDS = {"what", "is", "the", "of", "a", "an", "to", "in", "for", "and", "how"}

def mock_judge_relevance(query: str, chunk_text: str) -> str:
    query_terms = {w for w in query.lower().split() if w not in STOPWORDS and len(w) > 2}
    overlap = sum(1 for term in query_terms if term in chunk_text.lower())
    return "relevant" if overlap >= 1 else "irrelevant"

def mock_generate_answer(query: str, chunks: List[Dict]) -> str:
    relevant_text = " ".join(c["text"] for c in chunks)
    return f"Based on context: {relevant_text[:60]}..."

def mock_extract_claims(answer: str) -> List[str]:
    return [s.strip() for s in answer.split(".") if s.strip()]

def mock_check_claim_supported(claim: str, chunks: List[Dict]) -> bool:
    all_text = " ".join(c["text"] for c in chunks).lower()
    key_terms = [w for w in claim.lower().split() if len(w) > 4]
    return any(term in all_text for term in key_terms)

print("Mock judges defined")

Mock judges defined


In [6]:
def retrieve_node(state: RAGEvalState) -> RAGEvalState:
    # In a real pipeline this is a vector search call. Fixture data here for a
    # reproducible demo -- notice chunk_3 is deliberately off-topic.
    state["retrieved_chunks"] = [
        {"id": "c1", "text": "BST delete runs in O(h) time where h is tree height."},
        {"id": "c2", "text": "The successor node is the leftmost node of the right subtree."},
        {"id": "c3", "text": "Unrelated: Python lists are dynamic arrays."},
    ]
    return state


def judge_context_relevance_node(state: RAGEvalState) -> RAGEvalState:
    judgments = {c["id"]: mock_judge_relevance(state["query"], c["text"])
                 for c in state["retrieved_chunks"]}
    state["relevance_judgments"] = judgments
    relevant_count = sum(1 for v in judgments.values() if v == "relevant")
    state["context_relevance_score"] = relevant_count / len(judgments)
    return state


def generate_node(state: RAGEvalState) -> RAGEvalState:
    # Only feed relevant chunks into generation -- a common production pattern
    # (filter before you generate, don't just filter the eval after the fact).
    relevant_chunks = [c for c in state["retrieved_chunks"]
                        if state["relevance_judgments"][c["id"]] == "relevant"]
    state["answer"] = mock_generate_answer(state["query"], relevant_chunks)
    return state


def check_faithfulness_node(state: RAGEvalState) -> RAGEvalState:
    claims = mock_extract_claims(state["answer"])
    state["claims"] = claims
    scores = {c: mock_check_claim_supported(c, state["retrieved_chunks"]) for c in claims}
    state["faithfulness_scores"] = scores
    supported = sum(1 for v in scores.values() if v)
    state["faithfulness_score"] = supported / len(scores) if scores else 0.0
    return state

print("Nodes defined")

Nodes defined


In [7]:
builder = StateGraph(RAGEvalState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("judge_context_relevance", judge_context_relevance_node)
builder.add_node("generate", generate_node)
builder.add_node("check_faithfulness", check_faithfulness_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "judge_context_relevance")
builder.add_edge("judge_context_relevance", "generate")
builder.add_edge("generate", "check_faithfulness")
builder.add_edge("check_faithfulness", END)

rag_eval_graph = builder.compile()
print("Graph compiled: retrieve -> judge_context_relevance -> generate -> check_faithfulness -> END")

Graph compiled: retrieve -> judge_context_relevance -> generate -> check_faithfulness -> END


In [8]:
result = rag_eval_graph.invoke({"query": "What is the time complexity of BST delete?"})

print("Context relevance judgments:", result["relevance_judgments"])
print(f"Context relevance score:      {result['context_relevance_score']:.2f}")
print()
print("Answer:", result["answer"])
print("Faithfulness scores:", result["faithfulness_scores"])
print(f"Faithfulness score:           {result['faithfulness_score']:.2f}")

Context relevance judgments: {'c1': 'relevant', 'c2': 'irrelevant', 'c3': 'irrelevant'}
Context relevance score:      0.33

Answer: Based on context: BST delete runs in O(h) time where h is tree height....
Faithfulness scores: {'Based on context: BST delete runs in O(h) time where h is tree height': True}
Faithfulness score:           1.00


**Reading the output:**
- `c3` (Python lists — off-topic) is correctly flagged `irrelevant` → this is the **context relevance** signal, computed *before* generation even happens.
- The **faithfulness score** checks the *opposite direction*: given what got generated, is every claim traceable back to the retrieved context? A score of 1.00 here means no hallucination was detected — everything the model said maps to a retrieved chunk.
- These two scores catch **different failure modes**: bad context relevance = retriever problem. Bad faithfulness despite good context = generator problem (model ignored context / made things up).

---
# Part 3 — Agent Eval Pipeline as a LangGraph Graph

**Interview questions covered:**
- *How would you evaluate whether the agent selected the right tool?*
- *How would you measure task completion?*
- *What happens when the agent takes the wrong action?*
- *How would you evaluate multi-step tasks?*
- *How would you detect unnecessary tool calls?*

**Approach:** run an agent tool-calling loop (`plan_and_act`, looping via a conditional edge) that builds a **trace**, then hand the trace to a separate `evaluate` node. Keeping the agent loop and the eval logic as separate nodes mirrors real practice — the agent doesn't grade itself; a separate harness does, against a **gold trajectory** defined ahead of time.

In [9]:
class AgentEvalState(TypedDict):
    query: str
    gold_tool_sequence: List[str]     # ground truth trajectory, defined by the eval harness
    trace: List[Dict]                  # [{"tool", "args", "result", "used"}]
    current_step: int
    final_answer: Optional[str]
    task_success: Optional[bool]
    tool_selection_correct: List[bool]
    unnecessary_calls: List[str]

In [10]:
# ---- Mock tools the agent can call ----
def tool_search_docs(args):
    return f"docs about {args.get('topic', '?')}"

def tool_calculator(args):
    return str(eval(args.get("expr", "0")))  # toy only -- never eval() untrusted input in real code

def tool_send_email(args):
    return f"email sent to {args.get('to', '?')}"

TOOLS = {
    "search_docs": tool_search_docs,
    "calculator": tool_calculator,
    "send_email": tool_send_email,
}
print("Tools registered:", list(TOOLS.keys()))

Tools registered: ['search_docs', 'calculator', 'send_email']


In [11]:
# ---- Mock planner ----
# In production this is an LLM call deciding the next action given state + tool results.
# This plan deliberately repeats a call, to demonstrate "unnecessary tool call" detection.
def mock_plan_next_step(state: AgentEvalState) -> Optional[Dict]:
    plan = [
        {"tool": "search_docs", "args": {"topic": "BST delete complexity"}},
        {"tool": "search_docs", "args": {"topic": "BST delete complexity"}},  # redundant repeat
        {"tool": "calculator", "args": {"expr": "2**10"}},
    ]
    step = state["current_step"]
    return plan[step] if step < len(plan) else None

In [12]:
def plan_and_act_node(state: AgentEvalState) -> AgentEvalState:
    next_call = mock_plan_next_step(state)
    if next_call is None:
        state["final_answer"] = "Task complete."
        return state

    tool_name, args = next_call["tool"], next_call["args"]
    result = TOOLS[tool_name](args)

    # A call is "unnecessary" if the exact same (tool, args) already happened --
    # the result would be identical, so it added no new information.
    is_duplicate = any(t["tool"] == tool_name and t["args"] == args for t in state["trace"])

    state["trace"].append({
        "tool": tool_name, "args": args, "result": result, "used": not is_duplicate,
    })
    state["current_step"] += 1
    return state


def should_continue(state: AgentEvalState) -> str:
    # This is the loop: keep acting until the planner signals completion.
    return "plan_and_act" if state.get("final_answer") is None else "evaluate"


def evaluate_node(state: AgentEvalState) -> AgentEvalState:
    actual_tools = [t["tool"] for t in state["trace"]]
    gold_tools = state["gold_tool_sequence"]

    # Step-wise trajectory match against the gold sequence -- this is how you
    # evaluate MULTI-STEP tasks: score each step, not just the final answer.
    correctness = [
        (actual_tools[i] if i < len(actual_tools) else None) == gold_tools[i]
        for i in range(len(gold_tools))
    ]
    state["tool_selection_correct"] = correctness

    # Flags raised during the trace itself -- redundant calls with no new info.
    state["unnecessary_calls"] = [t["tool"] for t in state["trace"] if not t["used"]]

    # Task completion = reached a final answer AND every gold step was matched.
    # (In a real harness you'd also verify END STATE, e.g. did the email actually send.)
    state["task_success"] = (
        state["final_answer"] is not None and sum(correctness) == len(gold_tools)
    )
    return state

print("Nodes defined")

Nodes defined


In [13]:
builder = StateGraph(AgentEvalState)
builder.add_node("plan_and_act", plan_and_act_node)
builder.add_node("evaluate", evaluate_node)

builder.set_entry_point("plan_and_act")
builder.add_conditional_edges(
    "plan_and_act",
    should_continue,
    {"plan_and_act": "plan_and_act", "evaluate": "evaluate"},
)
builder.add_edge("evaluate", END)

agent_eval_graph = builder.compile()
print("Graph compiled: plan_and_act (loops) -> evaluate -> END")

Graph compiled: plan_and_act (loops) -> evaluate -> END


In [14]:
initial_state: AgentEvalState = {
    "query": "Look up BST delete complexity and compute 2^10",
    "gold_tool_sequence": ["search_docs", "calculator"],   # the IDEAL 2-step trajectory
    "trace": [],
    "current_step": 0,
    "final_answer": None,
    "task_success": None,
    "tool_selection_correct": [],
    "unnecessary_calls": [],
}

result = agent_eval_graph.invoke(initial_state)

print("=== Trace ===")
for i, step in enumerate(result["trace"]):
    print(f"  Step {i}: {step['tool']}({step['args']}) -> {step['result']!r}  | used={step['used']}")

print()
print("=== Eval ===")
print("Tool selection correctness vs gold:", result["tool_selection_correct"])
print("Unnecessary calls detected:        ", result["unnecessary_calls"])
print("Task success:                      ", result["task_success"])
print("Final answer:                      ", result["final_answer"])

=== Trace ===
  Step 0: search_docs({'topic': 'BST delete complexity'}) -> 'docs about BST delete complexity'  | used=True
  Step 1: search_docs({'topic': 'BST delete complexity'}) -> 'docs about BST delete complexity'  | used=False
  Step 2: calculator({'expr': '2**10'}) -> '1024'  | used=True

=== Eval ===
Tool selection correctness vs gold: [True, False]
Unnecessary calls detected:         ['search_docs']
Task success:                       False
Final answer:                       Task complete.


**Reading the output:**
- **Tool selection accuracy** — step 0 matches gold (`search_docs`), step 1 doesn't (gold expected `calculator`, agent repeated `search_docs`) → `[True, False]`. This is exactly the "right tool, wrong step" signal from the eval questions.
- **Unnecessary tool calls** — the duplicate `search_docs` call is flagged because its `(tool, args)` pair already appeared in the trace with no new information gained. In a real system you'd also track cost/latency here, since redundant calls are a common source of runaway agent cost.
- **Task completion** — `False`, because even though the agent reached a final answer, its trajectory deviated from the gold path at step 1. This shows why **task success ≠ "did it stop without error"** — you need the full trajectory check, not just presence of a final message.
- **What happens on a wrong action** — nothing in this toy agent *catches* the redundant call mid-loop; it just gets executed and flagged afterward by the eval harness. A more robust agent would validate before calling (e.g., check "have I already fetched this?") so the wrong action never happens, or self-correct by inspecting the trace so far. That's the gap between what this demo evaluates and what a production-grade agent would additionally prevent.

---
## Summary — mapping back to the interview answers

| Question | Where in this notebook |
|---|---|
| How would you evaluate retrieval quality? | Part 1 — full metric suite run against a labeled query |
| Precision@K / Recall@K | Part 1 — `precision_at_k`, `recall_at_k` |
| When MRR vs nDCG? | Part 1 — `mrr` (single best answer) vs `ndcg_at_k` (graded, multiple relevant) |
| Context relevance | Part 2 — `judge_context_relevance_node`, scored *before* generation |
| Groundedness / faithfulness | Part 2 — `check_faithfulness_node`, claim-by-claim check *after* generation |
| Right tool selected? | Part 3 — `tool_selection_correct`, step-wise match vs gold trajectory |
| Task completion | Part 3 — `task_success`, requires final answer AND full trajectory match |
| Wrong action taken | Part 3 — discussion above; this demo detects it post-hoc via eval, doesn't prevent it |
| Multi-step evaluation | Part 3 — step-wise trajectory scoring, not just final-answer scoring |
| Unnecessary tool calls | Part 3 — `unnecessary_calls`, duplicate `(tool, args)` detection |